# MLFP05 Quiz — Solutions (Instructor)

**This is the instructor answer key.** It is identical in structure to
`mlfp05_quiz.ipynb` but with every blank pre-filled and the Q5 TARGET
architecture wired in. Use it to:

1. Verify end-to-end runnability on Colab T4 (total runtime ≈ 8 minutes).
2. Confirm each `check_qN(...)` reports PASS on the canonical solutions.
3. Consult as a reference when grading student submissions.

---

## Quiz Structure — identical to the student notebook

| # | Skill                          | Dataset          | Threshold                          |
| - | ------------------------------ | ---------------- | ---------------------------------- |
| 1 | MLP from scratch               | MNIST            | test accuracy ≥ 0.95               |
| 2 | Convolutional classifier       | MNIST            | test accuracy ≥ 0.97               |
| 3 | LSTM on sequential signal      | synthetic halves | test accuracy ≥ 0.85               |
| 4 | Diagnose a broken CNN          | MNIST            | identify **stride + padding** bug  |
| 5 | Train-and-prescribe — iterate  | Fashion-MNIST    | 3 HEALTHY verdicts + test ≥ 0.82   |

All thresholds live in `quiz_harness.py` as named constants.

## Q5 Threshold Calibration Run (captured verdicts)

The `check_q5_pass` thresholds were set by running `Q5TargetMLP` on
Fashion-MNIST for 3 epochs, device=MPS (Apple Silicon), seed=0:

```
epoch 1/3  train_loss=0.9686  val_loss=0.5006
epoch 2/3  train_loss=0.4704  val_loss=0.4111
epoch 3/3  train_loss=0.3975  val_loss=0.3970
dead_neurons  : HEALTHY — worst layer net.1 at 0% inactive
loss_trend    : HEALTHY — train slope -2.86e-01/epoch
gradient_flow : CRITICAL (advisory) — max RMS 1.77e-02 on output layer
                (this is a 10-way softmax artifact, not a pathology — see slide 5F)
test_accuracy : 0.8666  (threshold 0.82)
```

The broken starter (`Q5BrokenStarter` with lr=0.1, no warmup, no weight
decay, ReLU + default init) produced:

```
dead_neurons  : WARNING — net.9 has 100% dead neurons
loss_trend    : HEALTHY (loss hits the 2.3 ceiling fast)
gradient_flow : CRITICAL
test_accuracy : 0.10   (random baseline)
```

So the starter fails on dead_neurons + test_acc, and the TARGET passes.
The gradient_flow verdict is printed advisory-only for both.

---


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Google Colab setup — one cell to rule them all.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
import os, sys

# ① EDIT THIS to point at YOUR fork of the Classroom repo.
#    Your fork URL is at the top of your assignment page on GitHub.
FORK_URL = "https://github.com/pcml-run26/pcml-run26-professional-certificate-in-machine-learning-pcml-run26-2601.git"
REPO_DIR = "/content/pcml-run26"

if not os.path.exists(REPO_DIR):
    !git clone {FORK_URL} {REPO_DIR}

# ② cd into the repo so relative data paths resolve
%cd {REPO_DIR}

# ③ Install deps — Colab has PyTorch preinstalled; add the Kailash stack.
!pip install -q kailash kailash-ml kailash-kaizen kailash-nexus \
               polars plotly gdown python-dotenv nest-asyncio

# ④ Make the `shared` package importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ⑤ Patch asyncio — Colab has a running event loop.
import nest_asyncio
nest_asyncio.apply()

# ⑥ GPU check — warn loudly if running on CPU.
import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU detected — training will be slow.')
    print('  Runtime → Change runtime type → T4 GPU (free tier)')

print('✓ Colab setup complete — shared.mlfp05 is importable')

In [ ]:
# Shared harness imports — used by every question.
import torch
import torch.nn as nn
import torch.nn.functional as F

from modules.mlfp05.quiz.quiz_harness import (
    pick_device,
    load_mnist,
    load_fashion_mnist,
    make_q3_dataset,
    train_classifier,
    train_and_diagnose,
    accuracy,
    check_q1, check_q2, check_q3, check_q4, check_q5_pass,
    Q4BrokenCNN, Q5TargetMLP, Q5BrokenStarter,
)

DEVICE = pick_device()
print(f'Using device: {DEVICE}')

---

## Q1 — Build a Multi-Layer Perceptron (10 min, ≥ 0.95 MNIST test accuracy)

Write an MLP with two hidden layers and train it for 5 epochs on MNIST.

**Spec**

* Input: flatten a 1×28×28 image to a 784-vector.
* Hidden: two `nn.Linear` layers with ReLU activations (sizes 256 and 128 work).
* Output: a `nn.Linear` head with 10 logits.
* Train: 5 epochs, Adam, lr=1e-3.

Grading cell calls `check_q1(final_test_accuracy)`. Threshold: **≥ 0.95**.


In [ ]:
# Q1 — student-editable model definition (SOLUTION)
class Q1MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# Q1 — train + grade (DO NOT EDIT)
torch.manual_seed(0)
q1_train, q1_test = load_mnist()
q1_acc = train_classifier(Q1MLP(), q1_train, q1_test, DEVICE, epochs=5, lr=1e-3, label='q1')
q1_pass, q1_score, q1_msg = check_q1(q1_acc)
print('\n' + q1_msg)

---

## Q2 — Build a Convolutional Classifier (15 min, ≥ 0.97 MNIST test accuracy)

Write a CNN that is strong enough to clear 0.97 in 3 epochs.

**Spec**

* Two `nn.Conv2d` blocks with `padding=1` and `nn.MaxPool2d(2)` between them.
* Two channel widths that roughly double: e.g. 32 → 64.
* A dense head: Flatten → Linear → ReLU → Linear(10).
* Train: 3 epochs, Adam, lr=1e-3.

Grading cell calls `check_q2(final_test_accuracy)`. Threshold: **≥ 0.97**.


In [ ]:
# Q2 — student-editable CNN (SOLUTION)
class Q2CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.head(self.conv(x))

In [ ]:
# Q2 — train + grade (DO NOT EDIT)
torch.manual_seed(0)
q2_train, q2_test = load_mnist()
q2_acc = train_classifier(Q2CNN(), q2_train, q2_test, DEVICE, epochs=3, lr=1e-3, label='q2')
q2_pass, q2_score, q2_msg = check_q2(q2_acc)
print('\n' + q2_msg)

---

## Q3 — Build an LSTM for a Sequential Signal (10 min, ≥ 0.85 halves-task accuracy)

The `make_q3_dataset()` helper produces a synthetic binary-sequence task: for each
length-20 Bernoulli sequence, the label is 1 iff more 1s are in the **first half**.
A bag-of-words MLP scores ~0.50; an LSTM that correctly reads the last hidden
state clears 0.85+.

**Spec**

* `nn.LSTM(input_size=1, hidden_size=32, batch_first=True)`
* `nn.Linear(32, 2)` head that classifies on the FINAL timestep hidden state.
* Train: 5 epochs, Adam, lr=1e-3.

Grading cell calls `check_q3(final_test_accuracy)`. Threshold: **≥ 0.85**.


In [ ]:
# Q3 — student-editable LSTM (SOLUTION)
class Q3LSTM(nn.Module):
    def __init__(self, hidden: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 2)
    def forward(self, x):
        h, _ = self.lstm(x)
        return self.head(h[:, -1, :])  # classify on final hidden state

In [ ]:
# Q3 — train + grade (DO NOT EDIT)
torch.manual_seed(0)
q3_train, q3_test = make_q3_dataset()
q3_acc = train_classifier(Q3LSTM(), q3_train, q3_test, DEVICE, epochs=5, lr=1e-3, label='q3')
q3_pass, q3_score, q3_msg = check_q3(q3_acc)
print('\n' + q3_msg)

---

## Q4 — Diagnose a Broken CNN (10 min, free-text answer)

We ship you a pre-built CNN (`Q4BrokenCNN`). The diagnostics report below
will surface an unhealthy gradient-flow pattern. Your job is to **diagnose**
the architectural bug and **prescribe** the fix in one short sentence.

Hints: the CNN has two `Conv2d` blocks starting from 28×28 MNIST input.
Think about what happens to the **spatial dimensions** as data flows through.

Grading cell calls `check_q4(MY_ANSWER)`. The checker accepts any phrasing
that mentions BOTH the diagnosed cause (stride / padding / dim collapse)
AND the fix (add padding or reduce stride).


In [ ]:
# Q4 — pre-built model + pre-built diagnostic run (DO NOT EDIT)
torch.manual_seed(0)
q4_train, q4_test = load_mnist()
q4_findings, q4_acc = train_and_diagnose(
    Q4BrokenCNN(),
    train_loader=q4_train, test_loader=q4_test, device=DEVICE,
    lr=1e-3, weight_decay=0.0, warmup_steps=0, epochs=3,
)
print('\nQ4 Diagnostics findings:')
for key, verdict in q4_findings.items():
    print(f'  {key:14s}: {verdict["severity"]:9s} — {verdict["message"]}')
print(f'\nQ4 broken CNN final test_acc: {q4_acc:.4f}')

In [ ]:
# Q4 — your answer (SOLUTION)
# Read the diagnostics output above and describe in one sentence:
#   (1) the architectural bug, and (2) the concrete fix.
MY_ANSWER = (
    'The CNN uses stride=2 with padding=0 in both Conv2d layers, which '
    'aggressively collapses the spatial dimensions and drives gradient flow '
    'into an unhealthy regime. Fix: add padding=1 (or reduce stride to 1) '
    'so the spatial dimensions do not shrink as fast.'
)
q4_pass, q4_msg = check_q4(MY_ANSWER)
print(q4_msg)

---

## Q5 — Train and Prescribe (30–45 min, iterate until PASS)

This is the capstone. Fashion-MNIST. Deep MLP. The harness does the
training AND the diagnostics; your job is to produce an architecture
and a hyperparameter configuration that clears all three diagnostic
gates plus the accuracy threshold.

**Passing criteria**

* `dead_neurons == HEALTHY` — no dead ReLUs.
* `loss_trend == HEALTHY` — loss converging, not oscillating.
* `test_acc >= 0.82` — Fashion-MNIST test accuracy after 3 epochs.
* `gradient_flow` is printed advisory-only (see slide 5F for why the
  library's 1e-2 RMS cut-off is biased by 10-way softmax output layers).

**Iteration loop**

1. Edit Cell 1 below (the model + hyperparameters).
2. Run Cell 2 (fixed scaffolding — DO NOT EDIT).
3. Read the Prescription Pad. If FAIL, find the unhealthy verdict and
   consult the deck slide it references (5C Stethoscope, 5F Blood Test,
   5H X-Ray).
4. Return to step 1 and iterate until PASS.

Re-running Cell 2 creates a NEW model and trains from scratch each time
— there is no hidden state between iterations.


In [ ]:
# Q5 Cell 1 — STUDENT EDITS (SOLUTION: the TARGET architecture)
#
# Target architecture diffs vs. the broken starter the student begins with:
#   • GELU activations instead of ReLU            (slide 5F — Blood Test)
#   • LayerNorm between hidden blocks              (slide 5H — X-Ray)
#   • Kaiming init on hidden Linear weights        (slide 5F)
#   • Xavier init on the output Linear             (to tame 10-way softmax grad RMS)
#   • Small non-zero bias everywhere (LN bias too) (keeps update_ratio finite)
#   • Dropout 0.1                                  (mild regularisation)
#   • LR 5e-4, weight decay 1e-4, warmup 300 steps (slide 5C — Stethoscope)

class MyModel(nn.Module):
    """Q5 TARGET — Fashion-MNIST 6-layer MLP with full prescription pad."""
    def __init__(self, dropout: float = 0.1):
        super().__init__()
        dims = [784, 512, 256, 128, 64, 32, 10]
        layers = [nn.Linear(dims[0], dims[1]), nn.GELU(), nn.Dropout(dropout)]
        for i, (in_dim, out_dim) in enumerate(zip(dims[1:-1], dims[2:])):
            layers.append(nn.LayerNorm(in_dim))
            layers.append(nn.Linear(in_dim, out_dim))
            if i < len(dims) - 3:
                layers.append(nn.GELU())
                layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
        # Init: Kaiming for hidden linears, Xavier for output, small non-zero
        # bias on every parameter (including LayerNorm bias) so the
        # diagnostics' update_ratio = ‖∇W‖/‖W‖ stays finite at step 0.
        linears = [m for m in self.modules() if isinstance(m, nn.Linear)]
        for m in linears[:-1]:
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.constant_(m.bias, 0.01)
        nn.init.xavier_normal_(linears[-1].weight)
        nn.init.constant_(linears[-1].bias, 0.01)
        for m in self.modules():
            if isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.constant_(m.bias, 0.01)
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

LR             = 5e-4
WEIGHT_DECAY   = 1e-4
WARMUP_STEPS   = 300

In [ ]:
# Q5 Cell 2 — train + diagnose + grade (DO NOT EDIT)
torch.manual_seed(0)
q5_train, q5_val = load_fashion_mnist()
q5_findings, q5_test_acc = train_and_diagnose(
    MyModel(),
    train_loader=q5_train, test_loader=q5_val, device=DEVICE,
    lr=LR, weight_decay=WEIGHT_DECAY, warmup_steps=WARMUP_STEPS, epochs=3,
)
print('\n' + '═' * 68)
print('  Q5 Prescription Pad')
print('═' * 68)
for key in ('gradient_flow', 'dead_neurons', 'loss_trend'):
    v = q5_findings.get(key, {'severity': 'UNKNOWN', 'message': ''})
    print(f'  {key:14s} : {v["severity"]:9s} — {v["message"]}')
print(f'  test_acc       : {q5_test_acc:.4f}')
print('═' * 68)
q5_pass, q5_msg = check_q5_pass(q5_findings, q5_test_acc)
print('\n' + q5_msg)

**If FAIL**: iterate on Cell 1 and re-run Cell 2. Consult the deck slide that
names the prescription for each unhealthy verdict:

| Verdict                 | Slide | Prescription                                        |
| ----------------------- | ----- | --------------------------------------------------- |
| `loss_trend` unhealthy  | 5C    | Lower LR, add warmup, add LayerNorm.                |
| `gradient_flow` CRITICAL| 5F    | Lower LR, add weight decay, switch to GELU, clip.  |
| `dead_neurons` WARNING  | 5H    | Switch to GELU (or LeakyReLU), use Kaiming init.    |
| `test_acc` low          | 5J    | Re-check all three above; widen hidden dims.         |

The TARGET in this notebook clears PASS in ~20 seconds on T4. Your mileage
may vary ±1% due to CUDA non-determinism.

---


## Submission Summary

This final cell tallies all five questions. Total score is printed inline
and survives the notebook save so instructors see it when grading.


In [ ]:
# Submission summary (DO NOT EDIT)
scores = [
    ('Q1 MLP',      q1_pass,  f'{q1_score:.4f}'),
    ('Q2 CNN',      q2_pass,  f'{q2_score:.4f}'),
    ('Q3 LSTM',     q3_pass,  f'{q3_score:.4f}'),
    ('Q4 Diagnose', q4_pass,  'free-text'),
    ('Q5 Prescribe', q5_pass, f'test_acc={q5_test_acc:.4f}'),
]
total = sum(int(p) for _, p, _ in scores)
print('\n' + '═' * 48)
print('  MLFP05 Quiz — submission summary')
print('═' * 48)
for name, passed, score in scores:
    marker = '✓' if passed else '✗'
    status = 'PASS' if passed else 'FAIL'
    print(f'  {marker} {name:14s}: {status}  ({score})')
print('─' * 48)
print(f'  TOTAL: {total}/5')
print('═' * 48)
if total == 5:
    print('  Full pass. Push this notebook to your fork to submit.')
elif total == 4:
    print('  Conditional pass. Review the one FAIL with your instructor.')
else:
    print('  Not yet passing. Fix the FAILs and re-run the affected cells.')